# Monthly Revenue Dashboard Analysis

## Project Overview
This notebook analyzes monthly revenue by product category, region, and customer segment using the Sample Superstore dataset. The focus is on trend analysis, contribution breakdown, variance analysis, and clear executive-style insights.

## Learning Objectives
- Build a monthly revenue dashboard workflow from transactional data.
- Break down revenue by product, region, and segment dimensions.
- Identify variance patterns: which months over/underperform expectations.
- Translate revenue patterns into actionable business insights.

## Problem Statement
Business leaders need a clear monthly view of revenue performance across product lines, geographies, and customer segments. Without this, planning, budgeting, and resource allocation stay reactive rather than proactive.

## Why This Project Matters
A well-structured revenue dashboard helps teams spot trends early, identify underperforming segments, and allocate resources to the highest-impact areas. It bridges the gap between raw transaction data and executive decision-making.

## Dataset Overview
This notebook uses the repo-local `Sample - Superstore.xls` workbook containing ~10K retail order lines with fields including `Order Date`, `Sales`, `Profit`, `Category`, `Sub-Category`, `Region`, `Segment`, `Quantity`, and `Discount`.

## Dataset Source and License Notes
The workbook is a widely circulated Sample Superstore training dataset already present in this repository. Confirm redistribution terms of the specific source copy you use outside this workspace.

## Environment Setup
Install required packages if not already available.

In [ ]:
import importlib, subprocess, sys

for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'xlrd']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Environment setup complete.')

## Imports

In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
np.random.seed(42)

## Configuration / Constants

In [ ]:
def locate_workspace_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'README.md').exists():
            return candidate
    return start_path

WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
DATASET_PATH = WORKSPACE_ROOT / 'Time Series Analysis' / 'Time Series Forecasting' / 'Sample - Superstore.xls'

EXPECTED_COLUMNS = [
    'Order Date', 'Sales', 'Profit', 'Quantity', 'Discount',
    'Category', 'Sub-Category', 'Region', 'Segment', 'Ship Mode'
]

print(json.dumps({'workspace_root': str(WORKSPACE_ROOT), 'dataset_path': str(DATASET_PATH)}, indent=2))

## Dataset Download and Loading
The dataset is loaded from a repo-local workbook. A clear error is raised if the file is missing.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Expected workbook not found at: {DATASET_PATH}')

raw_df = pd.read_excel(DATASET_PATH)
print(f'Raw shape: {raw_df.shape}')
raw_df.head()

## Data Validation Checks
Verify expected columns exist, check for nulls, duplicates, and date validity before analysis.

In [ ]:
missing_cols = sorted(set(EXPECTED_COLUMNS) - set(raw_df.columns))
if missing_cols:
    raise ValueError(f'Missing expected columns: {missing_cols}')

validation = pd.Series({
    'row_count': len(raw_df),
    'null_sales': int(raw_df['Sales'].isna().sum()),
    'null_order_date': int(raw_df['Order Date'].isna().sum()),
    'duplicate_rows': int(raw_df.duplicated().sum()),
    'unique_categories': int(raw_df['Category'].nunique()),
    'unique_regions': int(raw_df['Region'].nunique()),
    'unique_segments': int(raw_df['Segment'].nunique()),
}).to_frame('value')

print(validation.to_string())
print()
print(raw_df.isna().sum().to_frame('nulls').to_string())

## Data Cleaning and Preparation
Parse dates, remove duplicates, create time-based aggregation keys, and compute derived fields needed for revenue analysis.

In [ ]:
df = raw_df.drop_duplicates().copy()
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df = df.dropna(subset=['Order Date', 'Sales', 'Profit']).copy()

df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.month_name()
df['YearMonth'] = df['Order Date'].dt.to_period('M')
df['Quarter'] = df['Order Date'].dt.to_period('Q').astype(str)
df['Profit Margin %'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'] * 100, np.nan)

print(f'Clean rows: {len(df)}')
print(f'Date range: {df["Order Date"].min().date()} to {df["Order Date"].max().date()}')
df.head()

## Exploratory Data Analysis

### Overall Monthly Revenue Trend
Aggregate sales, profit, and order count by month to see the overall business trajectory.

In [ ]:
monthly = df.groupby('YearMonth', as_index=False).agg(
    revenue=('Sales', 'sum'),
    profit=('Profit', 'sum'),
    orders=('Order ID', 'nunique'),
    quantity=('Quantity', 'sum'),
    avg_discount=('Discount', 'mean'),
)
monthly['date'] = monthly['YearMonth'].dt.to_timestamp()
monthly['profit_margin_pct'] = np.where(monthly['revenue'] != 0,
                                         monthly['profit'] / monthly['revenue'] * 100, np.nan)
monthly['revenue_ma3'] = monthly['revenue'].rolling(3, min_periods=1).mean()
monthly['revenue_ma6'] = monthly['revenue'].rolling(6, min_periods=3).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].bar(monthly['date'], monthly['revenue'], color='steelblue', alpha=0.6, label='Monthly Revenue')
axes[0].plot(monthly['date'], monthly['revenue_ma3'], color='navy', linewidth=2, label='3-month MA')
axes[0].plot(monthly['date'], monthly['revenue_ma6'], color='darkorange', linewidth=2, label='6-month MA')
axes[0].set_title('Monthly Revenue with Moving Averages')
axes[0].set_ylabel('Revenue ($)')
axes[0].legend()

axes[1].plot(monthly['date'], monthly['profit_margin_pct'], marker='o', color='teal')
axes[1].axhline(monthly['profit_margin_pct'].mean(), color='red', linestyle='--', label='Avg margin')
axes[1].set_title('Monthly Profit Margin %')
axes[1].set_ylabel('Margin %')
axes[1].legend()

plt.tight_layout()
plt.savefig(Path(__file__).parent / 'monthly_revenue_trend.png' if '__file__' in dir() else 'monthly_revenue_trend.png', dpi=100, bbox_inches='tight')
plt.show()

monthly[['YearMonth', 'revenue', 'profit', 'profit_margin_pct', 'orders']].tail(12)

### Revenue by Product Category
Break down monthly revenue by product category to see which drives growth and which is volatile.

In [ ]:
cat_monthly = df.groupby(['YearMonth', 'Category'], as_index=False).agg(
    revenue=('Sales', 'sum'),
    profit=('Profit', 'sum'),
)
cat_monthly['date'] = cat_monthly['YearMonth'].dt.to_timestamp()

cat_total = df.groupby('Category', as_index=False).agg(
    total_revenue=('Sales', 'sum'),
    total_profit=('Profit', 'sum'),
).sort_values('total_revenue', ascending=False)
cat_total['revenue_share_pct'] = cat_total['total_revenue'] / cat_total['total_revenue'].sum() * 100
cat_total['profit_margin_pct'] = np.where(cat_total['total_revenue'] != 0,
                                           cat_total['total_profit'] / cat_total['total_revenue'] * 100, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for cat in cat_total['Category']:
    subset = cat_monthly[cat_monthly['Category'] == cat]
    axes[0].plot(subset['date'], subset['revenue'], marker='.', label=cat)
axes[0].set_title('Monthly Revenue by Category')
axes[0].legend()
axes[0].set_ylabel('Revenue ($)')

sns.barplot(data=cat_total, x='Category', y='revenue_share_pct', ax=axes[1], palette='Set2')
axes[1].set_title('Revenue Share by Category (%)')
axes[1].set_ylabel('Share %')

plt.tight_layout()
plt.show()

cat_total

### Revenue by Region
Identify geographic concentration and compare regional revenue trajectories over time.

In [ ]:
reg_monthly = df.groupby(['YearMonth', 'Region'], as_index=False).agg(
    revenue=('Sales', 'sum'),
    profit=('Profit', 'sum'),
)
reg_monthly['date'] = reg_monthly['YearMonth'].dt.to_timestamp()

reg_total = df.groupby('Region', as_index=False).agg(
    total_revenue=('Sales', 'sum'),
    total_profit=('Profit', 'sum'),
).sort_values('total_revenue', ascending=False)
reg_total['revenue_share_pct'] = reg_total['total_revenue'] / reg_total['total_revenue'].sum() * 100
reg_total['profit_margin_pct'] = np.where(reg_total['total_revenue'] != 0,
                                           reg_total['total_profit'] / reg_total['total_revenue'] * 100, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for reg in reg_total['Region']:
    subset = reg_monthly[reg_monthly['Region'] == reg]
    axes[0].plot(subset['date'], subset['revenue'], marker='.', label=reg)
axes[0].set_title('Monthly Revenue by Region')
axes[0].legend()
axes[0].set_ylabel('Revenue ($)')

sns.barplot(data=reg_total, x='Region', y='total_revenue', ax=axes[1], palette='rocket')
axes[1].set_title('Total Revenue by Region')
axes[1].set_ylabel('Revenue ($)')

plt.tight_layout()
plt.show()

reg_total

### Revenue by Customer Segment
Compare Consumer, Corporate, and Home Office segments by revenue contribution and profitability.

In [ ]:
seg_monthly = df.groupby(['YearMonth', 'Segment'], as_index=False).agg(
    revenue=('Sales', 'sum'),
    profit=('Profit', 'sum'),
)
seg_monthly['date'] = seg_monthly['YearMonth'].dt.to_timestamp()

seg_total = df.groupby('Segment', as_index=False).agg(
    total_revenue=('Sales', 'sum'),
    total_profit=('Profit', 'sum'),
    avg_discount=('Discount', 'mean'),
).sort_values('total_revenue', ascending=False)
seg_total['revenue_share_pct'] = seg_total['total_revenue'] / seg_total['total_revenue'].sum() * 100
seg_total['profit_margin_pct'] = np.where(seg_total['total_revenue'] != 0,
                                           seg_total['total_profit'] / seg_total['total_revenue'] * 100, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for seg in seg_total['Segment']:
    subset = seg_monthly[seg_monthly['Segment'] == seg]
    axes[0].plot(subset['date'], subset['revenue'], marker='.', label=seg)
axes[0].set_title('Monthly Revenue by Segment')
axes[0].legend()
axes[0].set_ylabel('Revenue ($)')

sns.barplot(data=seg_total, x='Segment', y='revenue_share_pct', ax=axes[1], palette='viridis')
axes[1].set_title('Revenue Share by Segment (%)')
axes[1].set_ylabel('Share %')

plt.tight_layout()
plt.show()

seg_total

## Variance Analysis
Compare each month's revenue against the rolling average to identify which months significantly over- or underperform expectations. This is the core of a revenue dashboard — spotting deviations early.

In [ ]:
monthly['variance_vs_ma6'] = monthly['revenue'] - monthly['revenue_ma6']
monthly['variance_pct'] = np.where(monthly['revenue_ma6'] != 0,
                                    monthly['variance_vs_ma6'] / monthly['revenue_ma6'] * 100, np.nan)

top_overperform = monthly.nlargest(5, 'variance_pct')[['YearMonth', 'revenue', 'revenue_ma6', 'variance_pct']]
top_underperform = monthly.nsmallest(5, 'variance_pct')[['YearMonth', 'revenue', 'revenue_ma6', 'variance_pct']]

colors = ['green' if v > 0 else 'firebrick' for v in monthly['variance_vs_ma6'].fillna(0)]

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(monthly['date'], monthly['variance_vs_ma6'], color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Monthly Revenue Variance vs 6-Month Moving Average')
ax.set_ylabel('Variance ($)')
plt.tight_layout()
plt.show()

print('Top overperforming months:')
print(top_overperform.to_string(index=False))
print()
print('Top underperforming months:')
print(top_underperform.to_string(index=False))

## Year-over-Year Comparison
Compare the same months across years to separate true growth from seasonality.

In [ ]:
yearly_monthly = df.groupby(['Year', 'Month'], as_index=False).agg(
    revenue=('Sales', 'sum'),
    profit=('Profit', 'sum'),
)

yoy_pivot = yearly_monthly.pivot(index='Month', columns='Year', values='revenue')

fig, ax = plt.subplots(figsize=(14, 5))
yoy_pivot.plot(kind='bar', ax=ax, width=0.75)
ax.set_title('Revenue by Month — Year over Year')
ax.set_ylabel('Revenue ($)')
ax.set_xlabel('Month')
ax.legend(title='Year')
plt.tight_layout()
plt.show()

annual = df.groupby('Year', as_index=False).agg(
    annual_revenue=('Sales', 'sum'),
    annual_profit=('Profit', 'sum'),
).sort_values('Year')
annual['yoy_growth_pct'] = annual['annual_revenue'].pct_change() * 100
annual

## Cross-Dimensional Revenue Heatmap
Combine category and region to spot dimension combinations that drive or drag revenue.

In [ ]:
cross = df.groupby(['Region', 'Category'], as_index=False).agg(
    revenue=('Sales', 'sum'),
    profit=('Profit', 'sum'),
)
cross_pivot_rev = cross.pivot(index='Region', columns='Category', values='revenue')
cross_pivot_profit = cross.pivot(index='Region', columns='Category', values='profit')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(cross_pivot_rev, annot=True, fmt=',.0f', cmap='YlGnBu', ax=axes[0])
axes[0].set_title('Revenue: Region x Category')

sns.heatmap(cross_pivot_profit, annot=True, fmt=',.0f', cmap='RdYlGn', ax=axes[1])
axes[1].set_title('Profit: Region x Category')

plt.tight_layout()
plt.show()

## Executive Summary Insights
Derive key business findings programmatically from the analysis above.

In [ ]:
best_month = monthly.loc[monthly['revenue'].idxmax()]
worst_month = monthly.loc[monthly['revenue'].idxmin()]
top_cat = cat_total.iloc[0]
top_reg = reg_total.iloc[0]
top_seg = seg_total.iloc[0]
overall_margin = df['Profit'].sum() / df['Sales'].sum() * 100

insights = [
    f'Total revenue across all years: ${df["Sales"].sum():,.0f}',
    f'Total profit: ${df["Profit"].sum():,.0f} (overall margin: {overall_margin:.1f}%)',
    f'Best revenue month: {best_month["YearMonth"]} with ${best_month["revenue"]:,.0f}',
    f'Worst revenue month: {worst_month["YearMonth"]} with ${worst_month["revenue"]:,.0f}',
    f'Top category by revenue: {top_cat["Category"]} ({top_cat["revenue_share_pct"]:.1f}% share)',
    f'Top region by revenue: {top_reg["Region"]} ({top_reg["revenue_share_pct"]:.1f}% share)',
    f'Top segment by revenue: {top_seg["Segment"]} ({top_seg["revenue_share_pct"]:.1f}% share)',
    f'Year-over-year growth (latest): {annual["yoy_growth_pct"].iloc[-1]:.1f}%',
]

for item in insights:
    print(f'  * {item}')

## Limitations
- This is a descriptive analysis of one sample dataset; patterns may not generalize.
- Revenue trends are not causally attributed — external factors like marketing spend, competition, and macroeconomics are absent.
- The dataset spans a limited time window, so long-term structural trends cannot be confirmed.
- Profit margin analysis depends on cost data quality in the workbook.

## Recommendations / Next Steps
- Investigate months with large negative variance to determine if they reflect seasonality, operational issues, or data gaps.
- Cross-reference regional underperformance with local market conditions and staffing levels.
- Build a forecasting model on top of this monthly revenue series for proactive planning.
- Add customer-level cohort analysis to separate new vs. repeat revenue contribution.

## Common Mistakes
- Comparing months without adjusting for seasonality or business calendar effects.
- Treating revenue growth as healthy without checking profit margin direction.
- Ignoring segment or regional concentration risk — one strong region can mask weakness elsewhere.
- Using absolute variance instead of percentage variance when comparing months of different scale.

## Mini Challenge
1. Add a sub-category breakdown to the variance analysis and find which sub-categories cause the biggest monthly swings.
2. Build a quarter-over-quarter growth table and identify which quarters consistently outperform.
3. Create a revenue-per-order metric by month and check whether order size is growing or shrinking.

## Final Summary / Key Takeaways
This notebook builds a monthly revenue dashboard workflow from raw retail transactions. It covers overall trend analysis, contribution breakdowns by product, region, and segment, variance analysis against rolling expectations, year-over-year comparisons, and cross-dimensional heatmaps. The key discipline: always compare revenue against profit and always check variance direction before drawing conclusions.